<a href="https://colab.research.google.com/github/barstow2/siop-26-ml-competition/blob/main/2026_SIOP_ML_Competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [58]:
# Imports
import os
import time
import json
import pathlib
from pathlib import Path
import pandas as pd
from google import genai
from google.colab import userdata
from google.colab import drive
from google.genai import types
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


The Colab instance needs to be in one of the countries listed here:  https://ai.google.dev/gemini-api/docs/available-regions

In [4]:
# Check the region that the Colab instance is in
!curl ipinfo.io

{
  "ip": "35.231.43.147",
  "hostname": "147.43.231.35.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}

Extraxct the API key from Colab secrets and set as an environment variable

In [5]:
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [6]:
# Initialize the genai client
# The client retrieves the API key from the env var automatically
client = genai.Client()

In [ ]:
print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

List of models that support generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
mode

### Early Testing

#### Test parsing a PDF file

In [ ]:
filepath = pathlib.Path("/content/drive/MyDrive/Data/study4.pdf")

In [ ]:
research_question = """
This meta-analysis examines how trust and subjective well-being are related:
do people who trust others and institutions more also tend to feel better about their lives?
"""

predictor_description = """
Trust: any measure capturing a respondent's belief, expectation, or confident assumption
that other people or human collectives (e.g., neighbours, strangers, organisations, institutions)
will act in a reliable, honest, fair, or benevolent manner.
Eligible labels include "trust," "confidence," or reverse-scored "distrust/mistrust."
Scales qualify if they target: (a) unspecified others (generalised/social trust),
(b) specific interpersonal partners (family, friends, romantic partners), or
(c) collective agents composed of/directed by/meant to serve people (government, courts, police, etc.).
EXCLUDE: self-trust, fate/technology trust, privacy concerns, compliance, satisfaction,
or attitudes toward policies — unless items explicitly assess perceived trustworthiness of human actors.
"""

dv_description = """
Subjective well-being: any measure indexing how satisfied, happy, or well a respondent feels
about life as a whole or their life circumstances.
Eligible labels include: life satisfaction, subjective well-being, happiness, positive affect,
quality of life, or reverse-scored: psychological distress, depression, anxiety, loneliness.
Scales qualify if they: (1) ask for a global life judgment, (2) aggregate affective frequency/intensity,
or (3) capture mental health symptoms treated as the negative pole of well-being.
EXCLUDE: physical-health-only indices or objective socioeconomic indicators (income, employment, education)
unless combined into a subjective life-evaluation score.
"""

prompt = f"""
You are a research assistant helping with a meta-analysis. Your task is to extract
bivariate effect sizes from this paper.

RESEARCH QUESTION: {research_question}

PREDICTOR (Trust): {predictor_description}

OUTCOME (Well-being): {dv_description}

INSTRUCTIONS:
1. Identify every variable in this paper that qualifies as a Trust measure (per the predictor definition).
2. Identify every variable that qualifies as a Well-being measure (per the outcome definition).
3. For each Trust × Well-being pair, extract the BIVARIATE Pearson's r from the correlation matrix or table.
   - Do NOT use standardized regression coefficients (β) from multivariate models if a bivariate r is available.
   - Only fall back to β, t, d, F, or other statistics if no bivariate r is reported.
4. If the well-being measure is reverse-coded (e.g., depression, distress, anxiety, loneliness),
   flip the sign so that a POSITIVE r means more trust = more well-being.
5. Report ONLY the effect sizes for qualifying Trust × Well-being pairs.
   Do not report relationships involving non-qualifying variables.

OUTPUT FORMAT — respond with ONLY valid JSON, no other text:
{{
  "trust_measures": ["list of identified trust measures"],
  "wellbeing_measures": ["list of identified well-being measures"],
  "effects": [
    {{
      "trust_measure": "name",
      "wellbeing_measure": "name",
      "statistic_type": "r",
      "value": 0.00,
      "reverse_coded_wellbeing": false,
      "value_after_sign_correction": 0.00,
      "location_in_paper": "e.g., Table 1"
    }}
  ]
}}
"""

In [ ]:
response_study4 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Part.from_bytes(
            data=filepath.read_bytes(),
            mime_type="application/pdf",
        ),
        prompt
    ]
)

print(response_study4.text)

#### Update the prompt and test with another PDF file

In [ ]:
filepath = pathlib.Path("/content/drive/MyDrive/Data/study17.pdf")

In [8]:
# Update the prompt
prompt = f"""
You are a research assistant helping with a meta-analysis. Your task is to extract
effect sizes from this paper.

RESEARCH QUESTION: {research_question}

PREDICTOR (Trust): {predictor_description}

OUTCOME (Well-being): {dv_description}

INSTRUCTIONS:
1. Identify every variable in this paper that qualifies as a Trust measure (per the predictor definition).
2. Identify every variable that qualifies as a Well-being measure (per the outcome definition).
3. For each Trust × Well-being pair, extract the effect size using this priority order:
   a. FIRST CHOICE: Bivariate Pearson's r from a correlation matrix or table.
   b. SECOND CHOICE: If no bivariate r is available, extract a test statistic (t, F with numerator df=1,
      or Cohen's d) along with the sample size or degrees of freedom.
   c. THIRD CHOICE: If neither of the above is available, extract the regression coefficient (B or β)
      from a regression model. Note whether it is standardized or unstandardized. If unstandardized,
      also extract the standard deviations of both the predictor and outcome variables if reported.
   d. If NONE of the above are available for a qualifying pair, do not include that pair.
4. If the well-being measure is reverse-coded (e.g., depression, distress, anxiety, loneliness),
   flip the sign so that a POSITIVE value means more trust = more well-being.
5. Report ONLY the effect sizes for qualifying Trust × Well-being pairs.
   Do not report relationships involving non-qualifying variables.
6. Extract only MAIN EFFECTS of the trust-wellbeing relationship.
   Do NOT decompose interaction terms to compute conditional effects for subgroups.
   If a regression table shows Trust × Moderator interaction terms, ignore them
   and extract only the main effect coefficient for the trust variable.

OUTPUT FORMAT — respond with ONLY valid JSON, no other text:
{{
  "trust_measures": ["list of identified trust measures"],
  "wellbeing_measures": ["list of identified well-being measures"],
  "effects": [
    {{
      "trust_measure": "name",
      "wellbeing_measure": "name",
      "statistic_type": "r | t | F | d | B_unstandardized | beta_standardized",
      "value": 0.00,
      "sample_size": null,
      "df": null,
      "sd_predictor": null,
      "sd_outcome": null,
      "reverse_coded_wellbeing": false,
      "value_after_sign_correction": 0.00,
      "location_in_paper": "e.g., Table 1",
      "notes": "e.g., from multivariate model controlling for covariates"
    }}
  ]
}}
"""

In [9]:
response_study17 = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Part.from_bytes(
            data=filepath.read_bytes(),
            mime_type="application/pdf",
        ),
        prompt
    ]
)

print(response_study17.text)

```json
{
  "trust_measures": [
    "Generalized trust"
  ],
  "wellbeing_measures": [
    "Life satisfaction",
    "Happiness"
  ],
  "effects": [
    {
      "trust_measure": "Generalized trust",
      "wellbeing_measure": "Life satisfaction",
      "statistic_type": "B_unstandardized",
      "value": 0.105,
      "sample_size": 37237,
      "df": null,
      "sd_predictor": 5.37,
      "sd_outcome": 2.08,
      "reverse_coded_wellbeing": false,
      "value_after_sign_correction": 0.105,
      "location_in_paper": "Table 3, Model A",
      "notes": "from multilevel linear regression model, controlling for gender, age, marital status, work status, problems in daily activities, and urbanization grade of residence"
    },
    {
      "trust_measure": "Generalized trust",
      "wellbeing_measure": "Happiness",
      "statistic_type": "B_unstandardized",
      "value": 0.072,
      "sample_size": 37237,
      "df": null,
      "sd_predictor": 5.37,
      "sd_outcome": 1.87,
      "rever

In [10]:
# Inspect tokens used to calculate costs before billing console
print(response_study4.usage_metadata)

cache_tokens_details=None cached_content_token_count=None candidates_token_count=133 candidates_tokens_details=None prompt_token_count=3591 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=753
), ModalityTokenCount(
  modality=<MediaModality.DOCUMENT: 'DOCUMENT'>,
  token_count=2838
)] thoughts_token_count=1070 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=4794 traffic_type=None


In [26]:
def get_cost_gemini_flash_25(response) -> dict[str, float]:
    """Calculate cost for a Gemini 2.5 Flash API response.

    Pricing (per 1M tokens):
        Input (text/image/video/document): $0.30
        Output (including thinking tokens): $2.50
    """
    INPUT_PRICE_PER_TOKEN = 0.30 / 1_000_000
    OUTPUT_PRICE_PER_TOKEN = 2.50 / 1_000_000

    meta = response.usage_metadata

    input_tokens = meta.prompt_token_count or 0
    thinking_tokens = meta.thoughts_token_count or 0
    candidate_tokens = meta.candidates_token_count or 0
    output_tokens = candidate_tokens + thinking_tokens

    input_cost = input_tokens * INPUT_PRICE_PER_TOKEN
    output_cost = output_tokens * OUTPUT_PRICE_PER_TOKEN
    total_cost = input_cost + output_cost

    return {
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "thinking_tokens": thinking_tokens,
        "input_cost": round(input_cost, 6),
        "output_cost": round(output_cost, 6),
        "total_cost": round(total_cost, 6),
    }

In [27]:
get_cost_gemini_flash_25(response_study4)

{'input_tokens': 3591,
 'output_tokens': 1203,
 'thinking_tokens': 1070,
 'input_cost': 0.001077,
 'output_cost': 0.003008,
 'total_cost': 0.004085}

- Function needs to take a pdf filename, research question, predictor description, and dependent variable description
- Output needs to be a csv with columns studyid and aggregateeffectsize

In [34]:
def parse_response(response_text: str) -> list[dict]:
    """Parse the JSON response from Gemini into a list of effect dicts."""
    cleaned = response_text.strip().removeprefix("```json").removesuffix("```").strip()
    parsed = json.loads(cleaned)
    return parsed["effects"]


def unstandardized_b_to_approx_r(b: float, sd_predictor: float, sd_outcome: float) -> float:
    """Convert unstandardized B to approximate standardized beta (proxy for r)."""
    if sd_predictor is None or sd_outcome is None:
        raise ValueError("Cannot convert unstandardized B without SDs for both variables.")
    return b * (sd_predictor / sd_outcome)


def effect_to_r(effect: dict) -> float:
    """Convert a single extracted effect to Pearson's r (or approximation)."""
    stat_type = effect["statistic_type"]
    value = effect["value_after_sign_correction"]

    if stat_type == "r":
        return value
    elif stat_type == "B_unstandardized":
        return unstandardized_b_to_approx_r(
            value, effect["sd_predictor"], effect["sd_outcome"]
        )
    elif stat_type == "beta_standardized":
        return value  # Already approximate r (biased if multivariate, but best we have)
    else:
        raise ValueError(f"Conversion not yet implemented for statistic_type: {stat_type}")


def aggregate_effect_size(response_text: str) -> float | None:
    """Parse response, convert all effects to r, return the mean."""
    effects = parse_response(response_text)
    if not effects:
        return None
    r_values = [effect_to_r(e) for e in effects]
    return sum(r_values) / len(r_values)

In [37]:
# Study 4 (Shao) - should give 0.34
print(aggregate_effect_size(response_study4.text))

0.34


In [38]:
# Study 17 (Valeeva) - should give ~0.239
print(aggregate_effect_size(response_study17.text))

0.2389205445290004


#### Try a full run

In [43]:
combined_prompt = f"""
You are a research assistant helping with a meta-analysis. Your task is to extract
effect sizes from this paper AND catalog what types of statistics are available.

RESEARCH QUESTION: {research_question}

PREDICTOR (Trust): {predictor_description}

OUTCOME (Well-being): {dv_description}

INSTRUCTIONS:
1. Identify every variable in this paper that qualifies as a Trust measure (per the predictor definition).
2. Identify every variable that qualifies as a Well-being measure (per the outcome definition).
3. For each qualifying Trust × Well-being pair, extract the effect size using this priority order:
   a. FIRST CHOICE: Bivariate Pearson's r from a correlation matrix or table.
   b. SECOND CHOICE: If no bivariate r is available, extract a test statistic (t, F with numerator df=1,
      or Cohen's d) along with the sample size or degrees of freedom.
   c. THIRD CHOICE: If neither of the above is available, extract the regression coefficient (B or β)
      from a regression model. Note whether it is standardized or unstandardized. If unstandardized,
      also extract the standard deviations of both the predictor and outcome variables if reported.
   d. If NONE of the above are available for a qualifying pair, do not include that pair in "effects"
      but still include it in "pairs_inventory".
4. If the well-being measure is reverse-coded (e.g., depression, distress, anxiety, loneliness),
   flip the sign so that a POSITIVE value means more trust = more well-being.
5. Report ONLY the effect sizes for qualifying Trust × Well-being pairs.
   Do not report relationships involving non-qualifying variables.
6. Extract only MAIN EFFECTS of the trust-wellbeing relationship.
   Do NOT decompose interaction terms to compute conditional effects for subgroups.

OUTPUT FORMAT — respond with ONLY valid JSON, no other text:
{{
  "trust_measures": ["list of identified trust measures"],
  "wellbeing_measures": ["list of identified well-being measures"],
  "pairs_inventory": [
    {{
      "trust_variable": "name",
      "wellbeing_variable": "name",
      "statistics_available": ["r", "beta_standardized", "B_unstandardized", "t", "F", "d", "OR", "chi_square", "other"],
      "sds_reported": true,
      "sample_size_reported": true,
      "notes": "any relevant context about what is available"
    }}
  ],
  "effects": [
    {{
      "trust_measure": "name",
      "wellbeing_measure": "name",
      "statistic_type": "r | t | F | d | B_unstandardized | beta_standardized",
      "value": 0.00,
      "sample_size": null,
      "df": null,
      "sd_predictor": null,
      "sd_outcome": null,
      "reverse_coded_wellbeing": false,
      "value_after_sign_correction": 0.00,
      "location_in_paper": "e.g., Table 1",
      "notes": "e.g., from multivariate model controlling for covariates"
    }}
  ]
}}
"""

In [ ]:
pdf_dir = pathlib.Path("/content/drive/MyDrive/Data")
study_ids = [f"study{i}" for i in range(1, 128)]

all_results = []
errors = []

for study_id in study_ids:
    filepath = pdf_dir / f"{study_id}.pdf"

    if not filepath.exists():
        print(f"SKIP: {study_id} — file not found")
        errors.append({"study_id": study_id, "error": "file not found"})
        continue

    print(f"Processing {study_id}...", end=" ")

    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Part.from_bytes(
                    data=filepath.read_bytes(),
                    mime_type="application/pdf",
                ),
                combined_prompt
            ]
        )

        cleaned = response.text.strip().removeprefix("```json").removesuffix("```").strip()
        parsed = json.loads(cleaned)
        parsed["study_id"] = study_id
        all_results.append(parsed)
        print("OK")

    except Exception as e:
        print(f"ERROR: {e}")
        errors.append({"study_id": study_id, "error": str(e)})

    time.sleep(0.5)

# Save results
with open("/content/drive/MyDrive/Outputs/combined_results.json", "w") as f:
    json.dump(all_results, f, indent=2)

if errors:
    with open("/content/drive/MyDrive/Outputs/combined_errors.json", "w") as f:
        json.dump(errors, f, indent=2)

print(f"\nDone. {len(all_results)} succeeded, {len(errors)} failed.")

#### Turn json file into output csv

In [51]:
with open("/content/drive/MyDrive/Outputs/combined_results.json", "r") as f:
    all_results = json.load(f)

rows = []
for result in all_results:
    study_id = result["study_id"]
    effects = result.get("effects", [])

    if not effects:
        rows.append({"studyid": study_id, "aggregateeffectsize": None})
        continue

    r_values = []
    for effect in effects:
        try:
            r_values.append(effect_to_r(effect))
        except Exception as e:
            print(f"WARNING: {study_id} — could not convert effect: {e}")
            continue

    if r_values:
        rows.append({"studyid": study_id, "aggregateeffectsize": sum(r_values) / len(r_values)})
    else:
        rows.append({"studyid": study_id, "aggregateeffectsize": None})

df = pd.DataFrame(rows).sort_values("studyid", key=lambda x: x.str.extract(r'(\d+)')[0].astype(int))

print(f"Total studies: {len(df)}")
print(f"Studies with effect sizes: {df['aggregateeffectsize'].notna().sum()}")
print(f"Studies missing effect sizes: {df['aggregateeffectsize'].isna().sum()}")

# Convert NAN values to 0
df["aggregateeffectsize"] = df["aggregateeffectsize"].fillna(0)

# Save as CSV in Drive
df.to_csv("/content/drive/MyDrive/Outputs/submission.csv", index=False)

Total studies: 127
Studies with effect sizes: 95
Studies missing effect sizes: 32


,studyid,aggregateeffectsize
0,study1,0.200000
1,study2,0.225000
2,study3,0.070904
3,study4,0.340000
4,study5,0.000000
5,study6,0.000000
6,study7,0.005000
7,study8,0.000000
8,study9,0.114000
9,study10,0.412500


In [56]:
zeros = df[df["aggregateeffectsize"] == 0]
print(zeros)

      studyid  aggregateeffectsize
4      study5                  0.0
5      study6                  0.0
7      study8                  0.0
13    study14                  0.0
20    study21                  0.0
24    study25                  0.0
26    study27                  0.0
27    study28                  0.0
33    study34                  0.0
35    study36                  0.0
36    study37                  0.0
38    study39                  0.0
40    study41                  0.0
41    study42                  0.0
48    study50                  0.0
55    study57                  0.0
56    study58                  0.0
63    study65                  0.0
67    study70                  0.0
72    study75                  0.0
75    study78                  0.0
81    study86                  0.0
87    study92                  0.0
90    study95                  0.0
96   study101                  0.0
98   study103                  0.0
99   study104                  0.0
103  study108       

### Updated Loop w/ retry logic and cost calculation

Todo:
- Improve the prompt via manual coding of the PDFs

In [ ]:
combined_prompt = f"""
You are a research assistant helping with a meta-analysis. Your task is to extract
effect sizes from this paper AND catalog what types of statistics are available.

RESEARCH QUESTION: {research_question}

PREDICTOR (Trust): {predictor_description}

OUTCOME (Well-being): {dv_description}

INSTRUCTIONS:
1. Identify every variable in this paper that qualifies as a Trust measure (per the predictor definition).
2. Identify every variable that qualifies as a Well-being measure (per the outcome definition).
3. For each qualifying Trust × Well-being pair, extract the effect size using this priority order:
   a. FIRST CHOICE: Bivariate Pearson's r from a correlation matrix or table.
   b. SECOND CHOICE: If no bivariate r is available, extract a test statistic (t, F with numerator df=1,
      or Cohen's d) along with the sample size or degrees of freedom.
   c. THIRD CHOICE: If neither of the above is available, extract the regression coefficient (B or β)
      from a regression model. Note whether it is standardized or unstandardized. If unstandardized,
      also extract the standard deviations of both the predictor and outcome variables if reported.
   d. If NONE of the above are available for a qualifying pair, do not include that pair in "effects"
      but still include it in "pairs_inventory".
4. If the well-being measure is reverse-coded (e.g., depression, distress, anxiety, loneliness),
   flip the sign so that a POSITIVE value means more trust = more well-being.
5. Report ONLY the effect sizes for qualifying Trust × Well-being pairs.
   Do not report relationships involving non-qualifying variables.
6. Extract only MAIN EFFECTS of the trust-wellbeing relationship.
   Do NOT decompose interaction terms to compute conditional effects for subgroups.

OUTPUT FORMAT — respond with ONLY valid JSON, no other text:
{{
  "trust_measures": ["list of identified trust measures"],
  "wellbeing_measures": ["list of identified well-being measures"],
  "pairs_inventory": [
    {{
      "trust_variable": "name",
      "wellbeing_variable": "name",
      "statistics_available": ["r", "beta_standardized", "B_unstandardized", "t", "F", "d", "OR", "chi_square", "other"],
      "sds_reported": true,
      "sample_size_reported": true,
      "notes": "any relevant context about what is available"
    }}
  ],
  "effects": [
    {{
      "trust_measure": "name",
      "wellbeing_measure": "name",
      "statistic_type": "r | t | F | d | B_unstandardized | beta_standardized",
      "value": 0.00,
      "sample_size": null,
      "df": null,
      "sd_predictor": null,
      "sd_outcome": null,
      "reverse_coded_wellbeing": false,
      "value_after_sign_correction": 0.00,
      "location_in_paper": "e.g., Table 1",
      "notes": "e.g., from multivariate model controlling for covariates"
    }}
  ]
}}
"""

#### Dynamically figure out how the filenames should be numbered

In [60]:
def get_next_version(directory: Path, prefix: str, extension: str) -> int:
    """Find existing files matching the pattern and return the next version number."""
    existing = list(directory.glob(f"{prefix}_v*.{extension}"))
    if not existing:
        return 2
    versions = []
    for f in existing:
        try:
            v = int(f.stem.split("_v")[-1])
            versions.append(v)
        except ValueError:
            continue
    return max(versions) + 1 if versions else 1


output_dir = Path("/content/drive/MyDrive/Outputs")

results_version = get_next_version(output_dir, "combined_results", "json")
errors_version = results_version
submission_version = get_next_version(output_dir, "submission", "csv")

results_path = output_dir / f"combined_results_v{results_version}.json"
errors_path = output_dir / f"combined_errors_v{errors_version}.json"
submission_path = output_dir / f"submission_v{submission_version}.csv"

print("Will save to:")
print(f"  {results_path}")
print(f"  {errors_path}")
print(f"  {submission_path}")

Will save to:
  /content/drive/MyDrive/Outputs/combined_results_v2.json
  /content/drive/MyDrive/Outputs/combined_errors_v2.json
  /content/drive/MyDrive/Outputs/submission_v2.csv


#### Run the loop that makes the API calls (with retry logic and cost calculation)

In [61]:
# Test with any one study
filepath = pdf_dir / "study1.pdf"
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Part.from_bytes(
            data=filepath.read_bytes(),
            mime_type="application/pdf",
        ),
        combined_prompt
    ]
)

print(get_cost_gemini_flash_25(response))

{'input_tokens': 8832, 'output_tokens': 2308, 'thinking_tokens': 1937, 'input_cost': 0.00265, 'output_cost': 0.00577, 'total_cost': 0.00842}


In [ ]:
pdf_dir = pathlib.Path("/content/drive/MyDrive/Data")
study_ids = [f"study{i}" for i in range(1, 128)]

all_results = []
errors = []
running_cost = 0.0

MAX_RETRIES = 3
RETRY_DELAY = 30  # seconds to wait after a 503

for study_id in study_ids:
    filepath = pdf_dir / f"{study_id}.pdf"

    if not filepath.exists():
        print(f"SKIP: {study_id} — file not found")
        errors.append({"study_id": study_id, "error": "file not found"})
        continue

    print(f"Processing {study_id}...", end=" ")

    success = False
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=[
                    types.Part.from_bytes(
                        data=filepath.read_bytes(),
                        mime_type="application/pdf",
                    ),
                    combined_prompt
                ]
            )

            cleaned = response.text.strip().removeprefix("```json").removesuffix("```").strip()
            parsed = json.loads(cleaned)
            parsed["study_id"] = study_id
            parsed["cost"] = get_cost_gemini_flash_25(response)
            all_results.append(parsed)

            study_cost = parsed["cost"].get("total_cost", 0)
            running_cost += study_cost
            print(f"OK — this: ${study_cost:.4f}, running total: ${running_cost:.4f}")

            success = True
            break

        except Exception as e:
            error_str = str(e)
            is_503 = "503" in error_str

            if is_503 and attempt < MAX_RETRIES:
                print(f"503 (attempt {attempt}/{MAX_RETRIES}), retrying in {RETRY_DELAY}s...", end=" ")
                time.sleep(RETRY_DELAY)
            else:
                print(f"ERROR: {e}")
                errors.append({"study_id": study_id, "error": error_str})
                break

    time.sleep(0.5)

# Save results
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2)

if errors:
    with open(errors_path, "w") as f:
        json.dump(errors, f, indent=2)

# Cost summary
total_cost = sum(r.get("cost", {}).get("total_cost", 0) for r in all_results)
total_input = sum(r.get("cost", {}).get("input_tokens", 0) for r in all_results)
total_output = sum(r.get("cost", {}).get("output_tokens", 0) for r in all_results)

print(f"\nDone. {len(all_results)} succeeded, {len(errors)} failed.")
print(f"Total input tokens: {total_input:,}")
print(f"Total output tokens: {total_output:,}")
print(f"Estimated total cost: ${total_cost:.4f}")

In [ ]:
with open(results_path, "r") as f:
    all_results = json.load(f)

rows = []
for result in all_results:
    study_id = result["study_id"]
    effects = result.get("effects", [])

    if not effects:
        rows.append({"studyid": study_id, "aggregateeffectsize": None})
        continue

    r_values = []
    for effect in effects:
        try:
            r_values.append(effect_to_r(effect))
        except Exception as e:
            print(f"WARNING: {study_id} — could not convert effect: {e}")
            continue

    if r_values:
        rows.append({"studyid": study_id, "aggregateeffectsize": sum(r_values) / len(r_values)})
    else:
        rows.append({"studyid": study_id, "aggregateeffectsize": None})

df = pd.DataFrame(rows).sort_values("studyid", key=lambda x: x.str.extract(r'(\d+)')[0].astype(int))

print(f"Total studies: {len(df)}")
print(f"Studies with effect sizes: {df['aggregateeffectsize'].notna().sum()}")
print(f"Studies missing effect sizes: {df['aggregateeffectsize'].isna().sum()}")

# Convert NAN values to 0
df["aggregateeffectsize"] = df["aggregateeffectsize"].fillna(0)

# Save as CSV in Drive
df.to_csv(submission_path, index=False)
